# Statistical Analysis of Delivery Operations

## Business Objective

This notebook evaluates whether the operational patterns identified during delivery analysis are statistically significant.

The analysis investigates associations between delivery performance and:

- Customer review score
- Delivery delay duration
- Processing time
- Freight cost
- Order complexity
- Geographic delivery performance

The objective is driver identification and evidence-based interpretation rather than causal inference.

## Important Principle

This is observational e-commerce data.

Therefore:

- Statistical significance does not prove causation.
- Association does not imply causal effect.
- Operational recommendations will be based on evidence of meaningful relationships, not causal claims.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from urllib.parse import quote_plus

from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [3]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = r"localhost\SQLEXPRESS"
database = "SupplyChainAnalytics"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" +
    quote_plus(connection_string)
)

In [4]:
query = """
SELECT *
FROM dbo.order_analytics
"""

df = pd.read_sql(query, engine)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 99441
Columns: 40


In [8]:
df.head()

,order_id,customer_id,customer_unique_id,order_status,customer_city,customer_state,customer_zip_code_prefix,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_count,product_count,seller_count,order_item_value,freight_value,total_order_value,payment_record_count,payment_type_count,total_payment_value,max_payment_installments,review_record_count,distinct_review_count,avg_review_score,min_review_score,max_review_score,processing_days,shipping_days,fulfillment_days,expected_delivery_days,delay_days,is_delivered,is_late,is_not_delivered,delivery_performance_category,delay_severity,has_multiple_items,has_multiple_sellers,order_value_bucket
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,delivered,campos dos goytacazes,RJ,28013,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,1.0000,1.0000,1.0000,58.9000,13.2900,72.1900,1.0000,1.0000,72.1900,2.0000,1.0000,1.0000,5.0000,5,5,0.0319,6.3674,7.6139,15.6257,-8.0118,1,0,0,Early,No Delay,0,0,Low Value
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,delivered,santa fe do sul,SP,15775,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,1.0000,1.0000,1.0000,239.9000,19.9300,259.8300,1.0000,1.0000,259.8300,3.0000,1.0000,1.0000,4.0000,4,4,0.0083,8.1458,16.2160,18.5465,-2.3306,1,0,0,Early,No Delay,0,0,Medium Value
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,delivered,para de minas,MG,35661,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,1.0000,1.0000,1.0000,199.0000,17.8700,216.8700,1.0000,1.0000,216.8700,5.0000,1.0000,1.0000,5.0000,5,5,0.0104,1.9083,7.9486,21.3938,-13.4451,1,0,0,Early,No Delay,0,0,Medium Value
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,delivered,atibaia,SP,12952,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,1.0000,1.0000,1.0000,12.9900,12.7900,25.7800,1.0000,1.0000,25.7800,2.0000,1.0000,1.0000,4.0000,4,4,0.0069,2.1375,6.1472,11.5833,-5.4361,1,0,0,Early,No Delay,0,0,Low Value
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,delivered,varzea paulista,SP,13226,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,1.0000,1.0000,1.0000,199.9000,18.1400,218.0400,1.0000,1.0000,218.0400,3.0000,1.0000,1.0000,5.0000,5,5,0.0090,11.8167,25.1146,40.4188,-15.3042,1,0,0,Early,No Delay,0,0,Medium Value


In [6]:
print("Total rows:", len(df))
print("Distinct orders:", df["order_id"].nunique())
print("Duplicate rows:", df["order_id"].duplicated().sum())

Total rows: 99441
Distinct orders: 99441
Duplicate rows: 0


In [7]:
delivered_df = df[
    df["is_delivered"] == 1
].copy()

late_df = delivered_df[
    delivered_df["is_late"] == 1
].copy()

ontime_df = delivered_df[
    delivered_df["is_late"] == 0
].copy()

print("Delivered:", len(delivered_df))
print("Late:", len(late_df))
print("On-Time/Early:", len(ontime_df))

Delivered: 96476
Late: 7827
On-Time/Early: 88649


## Statistical Framework

Each analysis follows the same structure:

1. Business Question
2. Business Hypothesis
3. Statistical Hypotheses
4. Appropriate Statistical Test
5. Result
6. Effect Size
7. Business Interpretation

We do not perform tests mechanically.

## Test 1: Review Score vs Delivery Performance

### Business Question

Do customers who receive late deliveries report lower review scores than customers whose orders arrive on time?

### Business Hypothesis

Late delivery is associated with poorer customer satisfaction.

### Statistical Hypotheses

**H₀:** Review score distributions are the same for late and on-time deliveries.

**H₁:** Review score distributions differ between late and on-time deliveries.

### Why Mann–Whitney U?

Review scores are ordinal values from 1 to 5 rather than continuous measurements.

Therefore Mann–Whitney U is more appropriate than relying on a normal-distribution assumption.

In [9]:
late_reviews = (
    late_df["avg_review_score"]
    .dropna()
)

ontime_reviews = (
    ontime_df["avg_review_score"]
    .dropna()
)

print("Late reviews:", len(late_reviews))
print("On-time reviews:", len(ontime_reviews))

Late reviews: 7662
On-time reviews: 88168


In [10]:
summary_reviews = pd.DataFrame({
    "Late": late_reviews.describe(),
    "On-Time": ontime_reviews.describe()
})

summary_reviews

,Late,On-Time
count,"7,662.0000","88,168.0000"
mean,2.5666,4.2942
std,1.6577,1.1462
min,1.0000,1.0000
25%,1.0000,4.0000
50%,2.0000,5.0000
75%,4.0000,5.0000
max,5.0000,5.0000


In [11]:
u_stat, p_value = stats.mannwhitneyu(
    late_reviews,
    ontime_reviews,
    alternative="two-sided"
)

print("Mann-Whitney U:", u_stat)
print("P-value:", p_value)

Mann-Whitney U: 150716905.5
P-value: 0.0


In [12]:
n1 = len(late_reviews)
n2 = len(ontime_reviews)

rank_biserial = 1 - (2*u_stat)/(n1*n2)

print("Rank-biserial correlation:", rank_biserial)

Rank-biserial correlation: 0.5537904846638264


## Test 2: Delay Duration vs Customer Satisfaction

### Business Question

As delivery delay becomes longer, do customer review scores tend to decrease?

### Hypothesis

Longer delays are associated with lower review scores.

### Statistical Test

Spearman correlation.

Why?

- Review score is ordinal.
- Delay may have a non-linear relationship with satisfaction.
- Spearman measures monotonic association rather than linear dependence.

In [13]:
delay_review = delivered_df[
    [
        "delay_days",
        "avg_review_score"
    ]
].dropna()

delay_review.head()

,delay_days,avg_review_score
0,-8.0118,5.0000
1,-2.3306,4.0000
2,-13.4451,5.0000
3,-5.4361,4.0000
4,-15.3042,5.0000


In [14]:
rho, p_value = stats.spearmanr(
    delay_review["delay_days"],
    delay_review["avg_review_score"]
)

print("Spearman correlation:", rho)
print("P-value:", p_value)

Spearman correlation: -0.17557033221723028
P-value: 0.0


## Test 3: Processing Time and SLA Failure

### Business Question

Do late orders spend more time in processing before carrier handoff?

### Operational Hypothesis

Longer seller processing time is associated with higher SLA failure.

### Statistical Hypotheses

**H₀:** Processing time distributions are equal.

**H₁:** Processing time differs between late and on-time deliveries.

### Test

Mann–Whitney U

In [15]:
late_processing = late_df["processing_days"].dropna()
ontime_processing = ontime_df["processing_days"].dropna()

print(len(late_processing))
print(len(ontime_processing))

7827
88635


In [16]:
processing_summary = pd.DataFrame({
    "Late": late_processing.describe(),
    "On-Time": ontime_processing.describe()
})

processing_summary

,Late,On-Time
count,"7,827.0000","88,635.0000"
mean,0.5130,0.4208
std,1.0855,0.8319
min,0.0000,0.0000
25%,0.0097,0.0090
50%,0.0174,0.0139
75%,0.8024,0.5854
max,30.8938,13.3139


In [17]:
u_stat, p_value = stats.mannwhitneyu(
    late_processing,
    ontime_processing,
    alternative="two-sided"
)

print("U statistic:", u_stat)
print("P-value:", p_value)

U statistic: 372984179.5
P-value: 1.9198138900444681e-28


In [18]:
n1 = len(late_processing)
n2 = len(ontime_processing)

processing_effect = 1 - (2*u_stat)/(n1*n2)

print("Rank-biserial:", processing_effect)

Rank-biserial: -0.07527568171207633


## Test 4: Freight Cost and Delivery Performance

### Business Question

Do late deliveries tend to involve different freight costs than successful deliveries?

### Hypothesis

Freight value differs between late and on-time operational segments.

### Test

Mann–Whitney U

In [19]:
late_freight = late_df["freight_value"].dropna()
ontime_freight = ontime_df["freight_value"].dropna()

freight_summary = pd.DataFrame({
    "Late": late_freight.describe(),
    "On-Time": ontime_freight.describe()
})

freight_summary

,Late,On-Time
count,"7,827.0000","88,649.0000"
mean,24.6223,22.6233
std,22.8323,21.4360
min,0.0000,0.0000
25%,15.1000,13.7200
50%,18.0500,17.0700
75%,26.4250,23.8500
max,711.3300,"1,794.9600"


In [20]:
u_stat, p_value = stats.mannwhitneyu(
    late_freight,
    ontime_freight,
    alternative="two-sided"
)

print("U statistic:", u_stat)
print("P-value:", p_value)

U statistic: 379084369.0
P-value: 3.253391103586339e-42


In [21]:
freight_effect = 1 - (
    2*u_stat /
    (len(late_freight)*len(ontime_freight))
)

print("Rank-biserial:", freight_effect)

Rank-biserial: -0.09268931979393646


## Test 5: Order Complexity and Late Delivery

### Business Question

Are operationally more complex orders more likely to be delivered late?

### Complexity Variable

Number of sellers involved in an order.

Orders are classified as:

- Single Seller
- Multi Seller

### Test

Chi-Square Test of Independence

In [24]:
delivered_df["seller_complexity"] = np.where(
    delivered_df["seller_count"] > 1,
    "Multi Seller",
    "Single Seller"
)

complexity_table = pd.crosstab(
    delivered_df["seller_complexity"],
    delivered_df["is_late"]
)

complexity_table

is_late,0,1
seller_complexity,,
Multi Seller,1257,18
Single Seller,87392,7809


In [25]:
chi2, p, dof, expected = stats.chi2_contingency(
    complexity_table
)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("P-value:", p)

Chi-square: 76.92295364601866
Degrees of freedom: 1
P-value: 1.7775908247710003e-18


In [26]:
n = complexity_table.values.sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(complexity_table.shape)-1))
)

print("Cramer's V:", cramers_v)

Cramer's V: 0.028236985539488033


## Test 6: Customer State and Late Delivery

### Business Question

Is late delivery associated with customer geography?

### Statistical Hypotheses

**H₀:** Late delivery is independent of customer state.

**H₁:** Late delivery and customer state are associated.

### Test

Chi-Square Test of Independence

Only states with sufficient operational volume are included.

In [27]:
state_counts = (
    delivered_df["customer_state"]
    .value_counts()
)

valid_states = state_counts[
    state_counts >= 100
].index

geo_df = delivered_df[
    delivered_df["customer_state"].isin(valid_states)
]

state_table = pd.crosstab(
    geo_df["customer_state"],
    geo_df["is_late"]
)

state_table.head()

is_late,0,1
customer_state,,
AL,302,95
AM,139,6
BA,2799,457
CE,1083,196
DF,1933,147


In [28]:
chi2, p, dof, expected = stats.chi2_contingency(
    state_table
)

print("Chi-square:", chi2)
print("P-value:", p)
print("Degrees of freedom:", dof)

Chi-square: 1617.0536779679674
P-value: 0.0
Degrees of freedom: 23


In [29]:
n = state_table.values.sum()

cramers_v_state = np.sqrt(
    chi2 / (n * (min(state_table.shape)-1))
)

print("Cramer's V:", cramers_v_state)

Cramer's V: 0.12959138614664722


## Test 7: Confidence Interval for Late Delivery Rate

The company observed a late delivery rate in this dataset.

We calculate a 95% confidence interval for the underlying late-delivery proportion among delivered orders.

In [30]:
late_orders = delivered_df["is_late"].sum()
total_delivered = len(delivered_df)

p_hat = late_orders / total_delivered

se = np.sqrt(
    p_hat * (1-p_hat) / total_delivered
)

ci_lower = p_hat - 1.96*se
ci_upper = p_hat + 1.96*se

print("Late Delivery Rate:", round(p_hat*100,2), "%")
print(
    "95% CI:",
    round(ci_lower*100,2),
    "% to",
    round(ci_upper*100,2),
    "%"
)

Late Delivery Rate: 8.11 %
95% CI: 7.94 % to 8.29 %


In [31]:
results = pd.DataFrame({
    "Business Question":[
        "Late vs On-Time Review Score",
        "Delay Duration vs Review Score",
        "Processing Time vs Lateness",
        "Freight Value vs Lateness",
        "Seller Complexity vs Lateness",
        "Customer State vs Lateness"
    ],
    "Statistical Test":[
        "Mann-Whitney U",
        "Spearman Correlation",
        "Mann-Whitney U",
        "Mann-Whitney U",
        "Chi-Square",
        "Chi-Square"
    ],
    "P-Value":[
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        np.nan
    ],
    "Effect Size":[
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        np.nan
    ]
})

results

,Business Question,Statistical Test,P-Value,Effect Size
0,Late vs On-Time Review Score,Mann-Whitney U,NaN,NaN
1,Delay Duration vs Review Score,Spearman Correlation,NaN,NaN
2,Processing Time vs Lateness,Mann-Whitney U,NaN,NaN
3,Freight Value vs Lateness,Mann-Whitney U,NaN,NaN
4,Seller Complexity vs Lateness,Chi-Square,NaN,NaN
5,Customer State vs Lateness,Chi-Square,NaN,NaN


# Statistical Findings and Business Interpretation

This notebook evaluated whether the operational patterns discovered during exploratory delivery analysis are statistically supported.

For every significant result, interpretation considers both:

- Statistical significance
- Practical significance using effect sizes

The analysis remains observational.

Therefore, identified relationships should be interpreted as operational associations rather than causal effects.

The next notebook will investigate multivariable relationships using regression models to identify which operational factors remain associated with delivery delay after controlling for other variables.